In [0]:
from pyspark.sql import functions as F

silver = "workspace.silver.adidas_us_sales"
dimension_gold = "workspace.gold.dim_calendario"
checkpoint = "/Volumes/workspace/gold/dim_calendario/checkpoint"

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.gold.dim_calendario")


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_examen")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dimension_gold} (
        id_calendario BIGINT, fecha DATE, ano INT, trimestre INT, mes INT,
        nombre_mes STRING, dia INT, nombre_dia STRING, semana_ano INT
    ) USING DELTA
""")

In [0]:
def cargar_dimension(lote, _):
    if lote.isEmpty():
        return

    meses = F.array(*[F.lit(mes) for mes in (
        "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
        "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre",
    )])
    dias = F.array(*[F.lit(dia) for dia in (
        "Domingo", "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado",
    )])
    anios = lote.select(F.year("fecha_factura").alias("ano")).distinct()
    fechas = anios.select(
        "ano", F.explode(F.sequence(F.make_date("ano", F.lit(1), F.lit(1)),
                                    F.make_date("ano", F.lit(12), F.lit(31)))).alias("fecha")
    )
    dimension = fechas.select(
        F.date_format("fecha", "yyyyMMdd").cast("long").alias("id_calendario"),
        "fecha", "ano", F.quarter("fecha").alias("trimestre"), F.month("fecha").alias("mes"),
        F.element_at(meses, F.month("fecha")).alias("nombre_mes"),
        F.dayofmonth("fecha").alias("dia"), F.element_at(dias, F.dayofweek("fecha")).alias("nombre_dia"),
        F.weekofyear("fecha").alias("semana_ano"),
    )
    dimension.createOrReplaceTempView("miembros_dimension")
    lote.sparkSession.sql(f"""
        MERGE INTO {dimension_gold} AS destino
        USING miembros_dimension AS origen
        ON destino.id_calendario = origen.id_calendario
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
flujo = (
    spark.readStream.option("readChangeFeed", "true").table(silver)
    .filter("_change_type IN ('insert', 'update_postimage')")
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .foreachBatch(cargar_dimension)
    .start()
)
flujo.awaitTermination()
